## Session 5 — Firestore + BigQuery

**XRO Tech · AI + Google Cloud Architecture Program · Batch 2**

Today your RAG app gains memory. Firestore stores conversation history so multi-turn chats work, and BigQuery logs every query for analytics.

Run every cell top to bottom. Read the markdown above each cell before running it.

**What you'll do:**

| Cell | Task |
|---|---|
| 1 | Check your environment |
| 2 | Connect to Firestore and test read/write |
| 3 | Build a RAG pipeline that uses conversation history |
| 4 | Set up BigQuery logging |
| 5 | Query your logs with SQL

**Cell 1 — Environment check.**

In [1]:
import os

STUDENT_ID = os.environ.get("STUDENT_ID", "NOT_SET")
BATCH_ID = os.environ.get("BATCH_ID", "NOT_SET")
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "NOT_SET")
REGION = "asia-south1"
GCS_BUCKET = "xro-lab-data"
MY_FOLDER = f"{BATCH_ID}/{STUDENT_ID}/"
BQ_DATASET = "xro_analytics"
BQ_TABLE = f"{PROJECT_ID}.{BQ_DATASET}.query_logs"

print(f"Student ID  : {STUDENT_ID}")
print(f"Batch ID    : {BATCH_ID}")
print(f"GCP Project : {PROJECT_ID}")
print(f"Region      : {REGION} (Mumbai)")
print(f"BQ Table    : {BQ_TABLE}")

Student ID  : s02
Batch ID    : batch2
GCP Project : xro-lab
Region      : asia-south1 (Mumbai)
BQ Table    : xro-lab.xro_analytics.query_logs


**Cell 2 — Connect to Firestore.** Firestore is a NoSQL document database. Each student's conversations are stored under their own document, keyed by student ID and session ID, so chat history stays separated by student.

If this cell fails with a permission error, stop here and check with your instructor before continuing — this is a one-time setup issue, not something wrong with your code.

In [2]:
from google.cloud import firestore
import time

db = firestore.Client(project=PROJECT_ID)


def save_message(session_id: str, role: str, content: str, tokens: int = 0):
    """Save one chat message to Firestore."""
    msg_ref = (
        db.collection("conversations")
        .document(f"{STUDENT_ID}_{session_id}")
        .collection("messages")
        .document()
    )
    msg_ref.set({
        "role": role,
        "content": content,
        "tokens": tokens,
        "timestamp": firestore.SERVER_TIMESTAMP,
        "student_id": STUDENT_ID,
    })
    return msg_ref.id


def get_history(session_id: str, last_n: int = 5) -> list:
    """Load the last N messages from a conversation, oldest first."""
    docs = (
        db.collection("conversations")
        .document(f"{STUDENT_ID}_{session_id}")
        .collection("messages")
        .order_by("timestamp", direction=firestore.Query.DESCENDING)
        .limit(last_n)
        .stream()
    )
    msgs = [{"role": d.get("role"), "content": d.get("content")} for d in docs]
    return list(reversed(msgs))


test_session_id = f"test_{int(time.time())}"
save_message(test_session_id, "user", "What is RAG?")
save_message(test_session_id, "assistant", "RAG stands for Retrieval Augmented Generation.")

history = get_history(test_session_id)
print(f"Firestore connected. Messages saved: {len(history)}")
for msg in history:
    print(f"  [{msg['role']:<10}] {msg['content'][:60]}")

Firestore connected. Messages saved: 2
  [user      ] What is RAG?
  [assistant ] RAG stands for Retrieval Augmented Generation.


**Cell 3 — RAG with memory.** This loads your Session 3 vector index, then answers questions using both the document context and the conversation history from Firestore. Ask a follow-up question and watch it understand "it" or "that" by reading the history.

`embed_text()` keeps the retry-on-rate-limit logic from Session 3, since this still calls the embedding API on every query.

In [3]:
import vertexai
import numpy as np
import json
from vertexai.language_models import TextEmbeddingModel
from vertexai.generative_models import GenerativeModel
from google.cloud import storage
from google.api_core.exceptions import ResourceExhausted

vertexai.init(project=PROJECT_ID, location=REGION)
embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-004")
generative_model = GenerativeModel("gemini-2.5-flash")

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(GCS_BUCKET)
blob = bucket.blob(f"{MY_FOLDER}session3_vector_index.json")
vector_index = json.loads(blob.download_as_text())["chunks"]


def embed_text(text: str, max_retries: int = 5) -> list[float]:
    """Convert text to a 768-dimension vector using Vertex AI, retrying on rate limits."""
    for attempt in range(max_retries):
        try:
            embeddings = embedding_model.get_embeddings([text])
            return embeddings[0].values
        except ResourceExhausted:
            wait_time = 2 ** attempt
            print(f"  Rate limit hit, waiting {wait_time}s before retry {attempt + 1}/{max_retries}...")
            time.sleep(wait_time)
    raise RuntimeError("Embedding failed after multiple retries. Wait a minute and try again.")


def cosine_sim(v1, v2):
    a, b = np.array(v1), np.array(v2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def rag_with_memory(question: str, session_id: str, top_k: int = 3) -> dict:
    """RAG query that uses Firestore chat history as conversational context."""
    t0 = time.time()

    qv = embed_text(question)
    chunks = sorted(vector_index, key=lambda c: cosine_sim(qv, c["vector"]), reverse=True)[:top_k]
    doc_context = "\n\n".join([f"[Doc chunk {i + 1}]\n{c['text']}" for i, c in enumerate(chunks)])

    history = get_history(session_id, last_n=5)
    history_text = "\n".join([f"{m['role'].upper()}: {m['content']}" for m in history])

    prompt = (
        "You are a helpful assistant. Use the document context and conversation history below.\n\n"
        f"DOCUMENT CONTEXT:\n{doc_context}\n\n"
        f"CONVERSATION HISTORY:\n{history_text}\n\n"
        f"CURRENT QUESTION: {question}\nANSWER:"
    )
    response = generative_model.generate_content(prompt)
    tokens = response.usage_metadata.total_token_count
    latency = round(time.time() - t0, 3)

    save_message(session_id, "user", question, 0)
    save_message(session_id, "assistant", response.text, tokens)

    return {"answer": response.text, "tokens": tokens, "latency_s": latency}


conv_id = f"s5demo_{int(time.time())}"
q1 = rag_with_memory("What is an embedding?", conv_id)
print(f"Q1 answer: {q1['answer'][:150]}")
time.sleep(0.5)

q2 = rag_with_memory("Can you give me a concrete example of what you just explained?", conv_id)
print(f"Q2 answer (uses history): {q2['answer'][:150]}")

print(f"\nHistory saved: {len(get_history(conv_id))} messages in Firestore")

/usr/local/lib/python3.11/dist-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
/usr/local/lib/python3.11/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Q1 answer: An embedding is a 768-dimensional vector that captures the semantic meaning of text. The text-embedding-004 model from Google converts any text into s
Q2 answer (uses history): For example, if you were to embed the text "a happy dog" and "a joyful puppy" using the text-embedding-004 model, their resulting 768-dimensional vect

History saved: 4 messages in Firestore


**Cell 4 — BigQuery logging.** Every RAG query gets logged to a BigQuery table: who asked, what they asked, how many tokens it used, and how long it took. This is the same kind of analytics pipeline a real product team would build.

This cell creates the dataset and table if they don't already exist, then logs the two test queries from Cell 3.

In [4]:
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID)

bq.create_dataset(BQ_DATASET, exists_ok=True)

schema = [
    bigquery.SchemaField("timestamp", "TIMESTAMP"),
    bigquery.SchemaField("student_id", "STRING"),
    bigquery.SchemaField("session_id", "STRING"),
    bigquery.SchemaField("question", "STRING"),
    bigquery.SchemaField("tokens_used", "INTEGER"),
    bigquery.SchemaField("latency_ms", "FLOAT"),
]
table = bigquery.Table(BQ_TABLE, schema=schema)
bq.create_table(table, exists_ok=True)
print(f"BigQuery table ready: {BQ_TABLE}")


def log_to_bq(session_id, question, tokens, latency_s):
    """Stream one row to BigQuery. No batching needed for this volume."""
    row = [{
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "student_id": STUDENT_ID,
        "session_id": session_id,
        "question": question[:500],
        "tokens_used": tokens,
        "latency_ms": round(latency_s * 1000, 1),
    }]
    errors = bq.insert_rows_json(BQ_TABLE, row)
    if errors:
        print(f"BQ insert error: {errors}")
    else:
        print(f"Logged: {question[:50]} -- {tokens} tokens, {round(latency_s * 1000)}ms")


log_to_bq(conv_id, "What is an embedding?", q1["tokens"], q1["latency_s"])
log_to_bq(conv_id, "Can you give me a concrete example?", q2["tokens"], q2["latency_s"])

BigQuery table ready: xro-lab.xro_analytics.query_logs
Logged: What is an embedding? -- 762 tokens, 4174ms
Logged: Can you give me a concrete example? -- 719 tokens, 2938ms


**Cell 5 — Query your logs with SQL.** BigQuery rows can take a minute or two to become queryable after a streaming insert. If this returns no rows immediately, wait a moment and run it again.

In [5]:
query = f"""
SELECT
  student_id,
  COUNT(*) AS total_queries,
  AVG(tokens_used) AS avg_tokens,
  AVG(latency_ms) AS avg_latency_ms,
  MAX(timestamp) AS last_query
FROM `{BQ_TABLE}`
WHERE student_id = '{STUDENT_ID}'
GROUP BY student_id
"""

print("Your query analytics")
results = bq.query(query).result()
for row in results:
    print(f"Student     : {row.student_id}")
    print(f"Queries     : {row.total_queries}")
    print(f"Avg tokens  : {row.avg_tokens:.0f}")
    print(f"Avg latency : {row.avg_latency_ms:.0f}ms")
    print(f"Last query  : {row.last_query}")

Your query analytics
Student     : s02
Queries     : 2
Avg tokens  : 740
Avg latency : 3556ms
Last query  : 2026-06-18 11:54:38+00:00


---

### Session 5 complete

Your RAG pipeline now remembers conversations across turns and logs every query for analytics — two things every production AI product needs.

**Before you close:** save the notebook, and commit to GitHub if you're tracking your work there.

**Next session:** Session 6 packages a RAG app with this same memory into a Docker container and deploys it as Project 2 on Cloud Run.